In [ ]:
import sys
!{sys.executable} -m pip install -q pandas==2.2.3 rapidfuzz==3.9.7 tqdm==4.66.5

In [ ]:
import os, re, time, hashlib, unicodedata
from datetime import datetime, timezone
import pandas as pd

def remove_accents(text):
    text='' if text is None else str(text)
    return ''.join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))
def normalize_text(text):
    t=remove_accents(text).lower(); t=re.sub(r'[^a-z0-9\s]',' ',t); return re.sub(r'\s+',' ',t).strip()
def normalize_name(name): return normalize_text(name)
def stable_hash(text): return hashlib.sha256(str(text).encode()).hexdigest()[:16]
def now_utc(): return datetime.now(timezone.utc).isoformat()
def save_csv(df,path): os.makedirs(os.path.dirname(path),exist_ok=True); df.to_csv(path,index=False,encoding='utf-8-sig')
for p in ['notebooks/input','notebooks/output','notebooks/raw','notebooks/cache','notebooks/reports']: os.makedirs(p,exist_ok=True)
path='notebooks/input/peps_input.csv'
if not os.path.exists(path):
    data=[['Claudia Sheinbaum Pardo','CDMX','Presidenta','Presidencia','Ejemplo técnico'],['Andrés Manuel López Obrador','Nacional','Expresidente','Presidencia','Ejemplo técnico'],['Marcelo Ebrard Casaubón','CDMX','Secretario','SRE','Ejemplo técnico'],['Adán Augusto López Hernández','Tabasco','Senador','Senado','Ejemplo técnico'],['Ricardo Monreal Ávila','Zacatecas','Senador','Senado','Ejemplo técnico'],['Xóchitl Gálvez Ruiz','Hidalgo','Senadora','Senado','Ejemplo técnico'],['Samuel García Sepúlveda','Nuevo León','Gobernador','Gobierno NL','Ejemplo técnico'],['Clara Brugada Molina','CDMX','Jefa de Gobierno','Gobierno CDMX','Ejemplo técnico'],['Omar García Harfuch','CDMX','Secretario','SSPC','Ejemplo técnico'],['Luisa María Alcalde Luján','CDMX','Dirigente','MORENA','Ejemplo técnico']]
    save_csv(pd.DataFrame(data,columns=['nombre','estado','puesto','dependencia','notas']),path)
df=pd.read_csv(path,dtype=str).fillna('')
for c in ['estado','puesto','dependencia','notas']:
    if c not in df.columns: df[c]=''
out=df.copy(); out['nombre_persona_input']=out['nombre']; out['normalized_name_input']=out['nombre'].map(normalize_name); out['token_sort_name']=out['normalized_name_input'].map(lambda x:' '.join(sorted(x.split()))); out['name_tokens']=out['normalized_name_input'].map(lambda x:'|'.join(x.split())); out['seed_id']=out['normalized_name_input'].map(stable_hash); out['estado_input']=out['estado']; out['puesto_input']=out['puesto']; out['dependencia_input']=out['dependencia']; out['notas_input']=out['notas']; out['has_only_name']=out[['estado_input','puesto_input','dependencia_input','notas_input']].eq('').all(axis=1); out['input_quality_score']=out['has_only_name'].map(lambda v:55 if v else 85); out['created_at']=now_utc()
save_csv(df,'notebooks/output/00_peps_input.csv')
cols=['seed_id','nombre_persona_input','normalized_name_input','token_sort_name','name_tokens','estado_input','puesto_input','dependencia_input','notas_input','has_only_name','input_quality_score','created_at']
save_csv(out[cols],'notebooks/output/00_peps_normalized.csv')
bench=[]
for n in [10,1000,10000]:
    s=time.perf_counter(); tmp=pd.concat([df[['nombre']]]*(n//len(df)+1),ignore_index=True).head(n); _=tmp['nombre'].map(normalize_name); t=time.perf_counter()-s; bench.append({'registros':n,'tiempo_normalizacion_segundos':t,'registros_por_segundo':n/t if t else 0})
save_csv(pd.DataFrame(bench),'notebooks/output/00_benchmark_normalizacion.csv')